# DEMO RANDOM FOREST

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
import pickle
import numpy as np
from PIL import Image, ImageOps
from collections import Counter
import sys
from types import ModuleType

In [ ]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None, gini=None, samples=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        self.gini = gini
        self.samples = samples
    def is_leaf_node(self):
        return self.value is not None

class DecisionTreeClassifier:
    def __init__(self, max_depth=10, min_samples_split=2, min_samples_leaf=1, max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.root = None
        self.n_features = None
        self.n_classes = None
        self.feature_importances_ = None
    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

class RandomForest:
    def __init__(self, n_estimators=100, max_depth=None, min_samples_split=2, max_features='sqrt', random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.trees = []
        self.random_state = random_state
        self.feature_importances_ = None
        self.n_features_ = None
        self.classes_ = None
        self.n_classes_ = None
    def predict(self, x):
        predictions = np.array([tree.predict(x) for tree in self.trees])
        y_pred = []
        for i in range(x.shape[0]):
            votes = predictions[:, i]
            y_pred.append(self._majority_vote(votes))
        return np.array(y_pred)
    def _majority_vote(self, votes):
        return Counter(votes).most_common(1)[0][0]

dt = ModuleType('decision_tree')
dt.Node = Node
dt.DecisionTreeClassifier = DecisionTreeClassifier
dt.RandomForest = RandomForest
sys.modules['decision_tree'] = dt

In [ ]:
with open('random_forest.pkl', 'rb') as f:
    model = pickle.load(f)
print(f'Model: {model.n_estimators} trees')

Model: 200 trees


In [ ]:
def extract(inp):
    if isinstance(inp, dict):
        return inp.get('composite') or inp.get('background') or (inp.get('layers') or [None])[0] or list(inp.values())[0]
    return inp

def test_all_methods(inp):
    img = extract(inp)
    if not img:
        return 'No image'
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img)
    if img.mode != 'L':
        img = img.convert('L')

    results = {}
    # Method 1: Raw pixels, WITH invert
    img_inv = ImageOps.invert(img)
    arr = np.array(img_inv.resize((28, 28), Image.Resampling.LANCZOS)).reshape(1, -1)
    results['1. Raw (0-255), WITH invert'] = int(model.predict(arr)[0])

    # Method 2: Normalize (0-1), no invert
    arr = np.array(img.resize((28, 28), Image.Resampling.LANCZOS)).astype(float) / 255.0
    arr = arr.reshape(1, -1)
    results['2. Normalize (0-1), NO invert'] = int(model.predict(arr)[0])

    return results

In [ ]:
def predict(inp):
    if not inp:
        return 'Draw a digit (BLACK or WHITE pen)'
    try:
        results = test_all_methods(inp)
        output = '# DEMO RANDOM FOREST:\n\n'
        for method, pred in results.items():
            output += f'**{method}:** {pred}\n\n'
        output += '---\n\nWhich one is CORRECT? Tell me!'
        return output
    except Exception as e:
        return f'Error: {e}'

In [ ]:
demo = gr.Interface(
    fn=predict,
    inputs=gr.Sketchpad(type='pil', image_mode='L', canvas_size=(280, 280), brush=gr.Brush(colors=['#000000', '#FFFFFF'], default_color='#FFFFFF', color_mode='select')),
    outputs=gr.Markdown(),
    title='DEMO RANDOM FOREST',
    description='Draw digit, see 2 different preprocessing results!'
)
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fbbcaf4a9601d8431c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
